In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

c:\Users\Ismat\anaconda3\envs\rec_env\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# OCR


In [2]:
# Install OCR Libraries
# Install Python wrapper
# Install pytesseract, pillow, and opencv-python
%pip install pytesseract pillow opencv-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pdf2image import convert_from_path

file_path1 = r"C:\Users\Ismat\OneDrive\Desktop\winter 24\Statistical Inference\Assignment2_STAT8430.pdf"

# Convert PDF to list of images
pages = convert_from_path(file_path1, dpi=300)  # dpi=300 gives good OCR quality
print(f"Converted PDF into {len(pages)} pages")

Converted PDF into 2 pages


In [5]:
import easyocr
from pdf2image import convert_from_path

# Convert PDF pages to images
file_path = r"C:\Users\Ismat\OneDrive\Desktop\winter 24\Statistical Inference\Assignment2_STAT8430.pdf"
pages = convert_from_path(file_path, dpi=300)

# Initialize EasyOCR Reader
reader = easyocr.Reader(['en'])  # specify language(s)

# OCR each page
all_text = ""
for i, page in enumerate(pages):
    # Convert PIL image to numpy array
    import numpy as np
    page_array = np.array(page)
    
    result = reader.readtext(page_array, detail=0)  # detail=0 returns text only
    page_text = " ".join(result)
    all_text += page_text + "\n"
    print(f"Page {i+1} OCR done")

# Preview extracted text
print(all_text[:1000])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Page 1 OCR done
Page 2 OCR done
Department of Math& Stat, University of Windsor Assignment Il: Stat. Inference (STAT8430) (Due date: March 4, 2024 at 2:30pm) Instructor: Dr. S. Nkurunziza, Ext. 3017 , Office 9-101 LT Instruction You are required to gives necessary details which show how the final solution was obtained and you are required to simplify as far as possible xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx Problem I. Let (02, &,P) be a probability space, let X be a random variable (r.V.) on (02,$,P) and let w be the characteristic function of X. Prove that p is uniformly continuous. Problem II: Let {Xn}o-1 be a sequence independent I.V. such that Xn Bin (1, 1 _ T+15 ) where 0 < a 1 and let Yn Ix;: j=1 1. Does Yn converge in mean square to 0, as n tends to infinity? Justify. 2. Does Yn converge probability to 0, as n tends to infinity? Justify: Problem III. Let {pn}0-1 be a sequence of real numbers such that lim Wi  n-on i=1 p € R: Let {Xn

In [6]:
# Extract OCR text
all_text = ""
for i, page in enumerate(pages):
    result = reader.readtext(np.array(page), detail=0)
    page_text = " ".join(result)
    all_text += page_text + "\n"

In [8]:
import re

def parse_questions(text):
    parsed = []
    
    # ---- Step 1: Clean OCR noise ----
    text = re.sub(r'\(cid:\d+\)', ' ', text)  # remove OCR artifacts
    text = re.sub(r'\*+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    # Remove header before first Problem
    text = re.split(r'Problem\s*I', text, flags=re.IGNORECASE)[-1]
    text = "Problem I " + text
    
    # Fix OCR mistakes
    text = text.replace("Problem Il", "Problem II")
    text = text.replace("Problem l", "Problem I")
    
    # ---- Step 2: Split Problems ----
    problems = re.split(r'Problem\s+[IVX]+\.?', text)
    
    for prob in problems:
        prob = prob.strip()
        if len(prob) < 50:
            continue
        
        # ---- Step 3: Split sub-questions (1,2,3) ----
        subqs = re.split(r'\d+\.', prob)
        
        for sub in subqs:
            sub = sub.strip()
            if len(sub) < 30:
                continue
            
            parsed.append({
                "question_text": sub,
                "options": [],
                "correct_answer": None
            })
    
    return parsed

parsed_questions = parse_questions(all_text)

print(f"Total questions: {len(parsed_questions)}")

print(json.dumps(parsed_questions[:5], indent=2))

Total questions: 4
[
  {
    "question_text": "Does X n converges in Lp to 0 as n tends to infinity? Justify: 2 . Does Xn converges in probability to 0 as n tends to infinity? Justify:",
    "options": [],
    "correct_answer": null
  },
  {
    "question_text": "Does Xn converges almost surely to 0 as n tends to infinity? Justify:",
    "options": [],
    "correct_answer": null
  },
  {
    "question_text": ": Let X be a random variable whose characteristic function is p.",
    "options": [],
    "correct_answer": null
  },
  {
    "question_text": "Suppose that ECIXI) O _ then E(X) = {v(0). 2 . Suppose that E(lXI?) 0, then E(X2) = #p\"(0).",
    "options": [],
    "correct_answer": null
  }
]


In [9]:
import json

# parsed_questions is your list of question dicts
output_file = "parsed_questions.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(parsed_questions, f, indent=2, ensure_ascii=False)

print(f"Saved {len(parsed_questions)} questions to {output_file}")

Saved 4 questions to parsed_questions.json
